# Football movement: completed Phases 1–4

This notebook runs intake → tracking → native-rate pose bouts → quality-gated measurements → decline analysis. Cached complete outputs are reused. The final study covers four 25-minute segments. No additional detector training occurs. See `FINAL_REPORT.md` and `LIMITATIONS.md`; movement decline is not a physiological fatigue diagnosis.


In [ ]:
from pathlib import Path
import sys, json
ROOT = Path.cwd()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
assert (ROOT / 'src').is_dir(), 'Open this notebook from the project directory'
sys.path.insert(0, str(ROOT))
from src.common import load_config
CONFIG = ROOT / 'configs/study_117093_first.yaml'
config = load_config(CONFIG)
config['pipeline']['targets'] = [{'player_id': 'A_blue_36', 'reference_player_id': '473125'}]
config['pipeline']['pose_mode'] = 'running_bouts'
config['pilot']['pose_model'] = 'yolov8m-pose.pt'
TRACKS = config['output_dir']
POSES = config['pipeline'].get('pose_output_dir', ROOT / 'results/phase3/study/117093_first')
METRICS = ROOT / 'results/metrics_csv/117093_first'


## Environment and input

Install once outside the notebook: `uv sync --locked --group dev --group notebook`. Authenticate using `uv run --no-sync hf auth login` and the dataset's accepted access conditions. Videos are in `data/hub/`, preserving upstream filenames. A fresh machine needs the files listed by the checked download manifests. Initial ID selection and field correspondences are configurable; the study uses supplied annotations to assist identity only.


In [ ]:
from src.environment import check_environment
report_path = ROOT / 'results/phase1/environment.json'
environment = json.loads(report_path.read_text()) if report_path.exists() else check_environment(report_path, True, 'cuda:0')
assert environment['status'] == 'passed'
from src.video_io import probe_video
probe_video(config['video']['path'])


## Audited detector and calibration

One 15-epoch detector experiment was run. Validation selected the original YOLOv8n. A direct homography fails on the curved panoramic pixel space; a quadratic rational ground warp passed a withheld field-point check. Ordinary planar views can use `pixel_to_world` with a validated homography. Elevated hips remain image-space joints.


In [ ]:
training = json.loads((ROOT / 'results/phase2/training/bounded_run.json').read_text())
assert training['status'] == 'completed'
from src.calibrate import GroundMapping
mapping = GroundMapping.load(config['pipeline']['ground_mapping'])
assert mapping.validated
{'chosen_detector': training['selection'], 'field_method': mapping.method, 'validation': mapping.payload['validation']}


## Track selected player

Tracklets, ambiguous links, camera checks and gaps are saved separately. The stored first-half run includes an extra early-only pilot player; it is excluded from this cohort before pose extraction. Existing partial runs are never treated as complete.


In [ ]:
from src.detect_track import extract_segment
if not (TRACKS / 'extraction_report.json').exists():
    extract_segment(config)
import pandas as pd
tracks = pd.read_csv(TRACKS / 'keypoints_positions.csv')
tracks = tracks[tracks.player_id == 'A_blue_36']
POSES.mkdir(parents=True, exist_ok=True)
selected = POSES / 'selected_tracks.csv'
tracks.to_csv(selected, index=False)
tracks.groupby('observation_status').size()


## Native 25-fps pose extraction

Only eligible running bouts spend pose compute. Missing/occluded joints remain missing; each crop origin is restored to the original source coordinates. Confidence is a gate, not anatomical ground truth. Gait cycles and vertical motion remain 2D proxies.


In [ ]:
from src.pose_pass import pose_from_tracks
if not (POSES / 'pose_report.json').exists():
    pose_from_tracks(config, selected, POSES, batch_size=8)
json.loads((POSES / 'pose_report.json').read_text())


## Movement, cycles, windows and plots

120-second windows need 70% ground-position exposure for movement. Gait needs 20 accepted running seconds and ten same-leg cycles. Ineligible values are NaN. Display windows overlap; separate non-overlapping windows support later inference. Compare speed, source body scale and projected image-motion strata before discussing mechanics.


In [ ]:
from src.process_metrics import process_metrics
source = POSES / 'keypoints_positions_camera_qualified.csv'
if not source.exists(): source = POSES / 'keypoints_positions.csv'
measurements = process_metrics(source, METRICS)
windows = pd.read_csv(METRICS / 'windowed_metrics_nonoverlap.csv')
display(windows)
measurements


In [ ]:
from IPython.display import Image, display
for name in ['avg_speed', 'stride_frequency', 'stride_length', 'knee_angle_range']:
    display(Image(filename=str(METRICS / 'plots' / f'{name}.png')))


## Full study and decline analysis

`uv run --no-sync python scripts/run_study.py` processes/resumes all four 25-minute segments. `uv run --no-sync python scripts/finalize_phase3.py` qualifies camera registration and combines the cohort tables. The following stage uses these completed tables. Missing baseline exposure, gait or between-half periods cannot become fabricated fatigue timestamps.


In [ ]:
from src.fatigue_analysis import run_analysis
ANALYSIS = ROOT / 'results/phase4'
STUDY = ROOT / 'data/manifests/study.json'
TABLES = ROOT / 'results/metrics_csv'
if not (TABLES / 'all_windowed_metrics_nonoverlap.csv').exists():
    # A fresh one-clip demo can finish without requiring the other three clips.
    import shutil
    from src.common import write_json
    TABLES = ANALYSIS / 'single_clip_input'
    TABLES.mkdir(parents=True, exist_ok=True)
    manifest = json.loads(STUDY.read_text())
    manifest['clips'] = [c for c in manifest['clips'] if c['clip_name'] == windows.clip_name.iloc[0]]
    STUDY = TABLES / 'study.json'
    write_json(STUDY, manifest)
    windows.to_csv(TABLES / 'all_windowed_metrics_nonoverlap.csv', index=False)
    shutil.copytree(METRICS, TABLES / windows.clip_name.iloc[0], dirs_exist_ok=True)
    frame_data = pd.read_parquet(METRICS / 'frame_metrics.parquet')
    rows = []
    for player, part in frame_data.groupby('player_id'):
        rows.append({'player_id': player, 'half': config['video']['half'], 'valid_movement_seconds': part.valid_dt.sum(), 'movement_frame_coverage': part.movement_valid.mean()})
    pd.DataFrame(rows).to_csv(TABLES / 'study_coverage.csv', index=False)
    ANALYSIS = ANALYSIS / 'single_clip_demo'
final_report = run_analysis(STUDY, TABLES, ANALYSIS)
display(pd.read_csv(ANALYSIS / 'analysis_status.csv'))
display(pd.read_csv(ANALYSIS / 'early_late_comparison.csv'))


In [ ]:
for name in ['avg_speed', 'peak_speed', 'p90_running_speed', 'high_intensity_distance_per_minute']:
    display(Image(filename=str(ANALYSIS / 'plots' / f'{name}_decline.png')))
from IPython.display import FileLink
display(FileLink(str(ANALYSIS / 'dashboard.html')))
json.loads((ANALYSIS / 'synthetic_controls.json').read_text())


## Independent broadcast robustness and annotated evidence

Five short public SoccerNet-GSR test clips are sampled at about 2 fps to audit detection, pose availability and camera motion. These samples cannot support native-rate cadence or long-term fatigue. `uv run --no-sync python -m src.broadcast_audit` reruns the separate audit; `uv run --no-sync python scripts/create_annotated_demo.py` renders a continuous native-rate study excerpt from saved joints without additional model inference.


In [ ]:
broadcast = ANALYSIS / 'broadcast_audit/report.json'
if broadcast.exists():
    display(pd.read_csv(ANALYSIS / 'broadcast_audit/clips.csv'))
else:
    print('Run the optional broadcast audit command above for this separate stress test.')
from IPython.display import Video
preview = ANALYSIS / 'annotated_demo.mp4'
if preview.exists(): display(Video(str(preview), embed=False))
